In [1]:
from scapy.all import conf, sr1, IP, ICMP

def get_gateway_ip_address():
    ip_of_gateway = conf.route.route("0.0.0.0")[2]
    return ip_of_gateway

if __name__ == "__main__":
    gateway_ip = get_gateway_ip_address()
    print("Gateway IP of your network device is : " + gateway_ip)

Gateway IP of your network device is : 192.168.209.171


In [5]:
import subprocess
import platform
import random
import socket
import time
import ipaddress

class NetworkDiscovery:
    def __init__(self):
        self.random_delays = [0.05, 0.1, 0.2, 0.3]
        self.timeout_range = [0.5, 1, 1.5]

    def _generate_random_ping_params(self):
        return {
            'count': random.randint(1, 2),
            'timeout': random.choice(self.timeout_range),
            'delay': random.choice(self.random_delays)
        }

    def ping_ip(self, ip):
        params = self._generate_random_ping_params()
        
        if platform.system().lower() == 'windows':
            command = [
                'ping', 
                '-n', str(params['count']), 
                '-w', str(int(params['timeout'] * 1000)), 
                ip
            ]
        else:
            command = [
                'ping', 
                '-c', str(params['count']), 
                '-W', str(params['timeout']), 
                ip
            ]
        
        try:
            result = subprocess.run(
                command, 
                stdout=subprocess.PIPE, 
                stderr=subprocess.PIPE, 
                timeout=params['timeout'] + 1
            )
            time.sleep(params['delay'])
            return result.returncode == 0
        except subprocess.TimeoutExpired:
            return False

    def get_network_range(self):
        try:
            s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
            s.connect(("8.8.8.8", 80))
            local_ip = s.getsockname()[0]
            s.close()
            
            network = ipaddress.ip_network(f"{local_ip}/24", strict=False)
            return str(network.network_address).split('.')[:-1]
        except Exception:
            return None

    def scan_network(self):
        network_base = self.get_network_range()
        if not network_base:
            return []

        network_base = '.'.join(network_base)
        active_devices = []

        ip_range = list(range(1, 255))
        random.shuffle(ip_range)

        for i in ip_range:
            ip = f"{network_base}.{i}"
            if self.ping_ip(ip):
                active_devices.append(ip)
            
            time.sleep(random.uniform(0.01, 0.1))

        return active_devices

def main():
    discovery = NetworkDiscovery()
    print("Initiating network discovery...")
    devices = discovery.scan_network()
    
    if devices:
        print("Discovered active network devices:")
        for device in devices:
            print(f"  - {device}")
    else:
        print("No devices found.")

if __name__ == "__main__":
    main()

Initiating network discovery...


Discovered active network devices:
  - 192.168.209.90
  - 192.168.209.171
